# Download BTS On-Time Performance Data (2024)

This notebook downloads monthly BTS On-Time data for 2024 and stores each month as parquet.

## Outputs
- `data/raw/bts/flights_2024_01.parquet` ... `data/raw/bts/flights_2024_12.parquet`
- `data/raw/bts/monthly_counts_2024.csv`

## Notes
- Designed to be re-runnable (skips months that already exist)
- Fails loudly at the end if any month is missing
- Works in local and Colab environments via environment variables

In [2]:
import io
import os
import zipfile
from pathlib import Path

import pandas as pd
import requests
from tqdm.notebook import tqdm

In [ ]:
# BTS download URL pattern
BASE_URL = "https://transtats.bts.gov/PREZIP/On_Time_Reporting_Carrier_On_Time_Performance_1987_present_{year}_{month}.zip"

YEAR = 2024

def resolve_project_root() -> Path:
    env_root = os.getenv("FLIGHT_PROJECT_ROOT")
    if env_root:
        return Path(env_root).expanduser().resolve()

    cwd = Path.cwd().resolve()
    for p in [cwd] + list(cwd.parents):
        if (p / "data").exists() and (p / "notebooks").exists():
            return p
    return cwd

PROJECT_ROOT = resolve_project_root()
DATA_ROOT = Path(os.getenv("FLIGHT_DATA_DIR", PROJECT_ROOT / "data")).expanduser().resolve()
RAW_DIR = DATA_ROOT / "raw" / "bts"
RAW_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Data root:    {DATA_ROOT}")
print(f"Raw dir:      {RAW_DIR}")

Data root: /Users/hazel/Desktop/26_spring/cis_5450/CIS-5450/data
Raw dir:   /Users/hazel/Desktop/26_spring/cis_5450/CIS-5450/data/raw


In [4]:
total_rows = 0
monthly_counts = []
failed_months = []

for month in tqdm(range(1, 13), desc="Overall progress"):
    out_path = RAW_DIR / f"flights_{YEAR}_{month:02d}.parquet"

    # Re-runnable: if parquet already exists, count rows and skip downloading.
    if out_path.exists():
        try:
            rows = len(pd.read_parquet(out_path, columns=["FlightDate"]))
            total_rows += rows
            monthly_counts.append({"year": YEAR, "month": month, "rows": rows, "source": "cached"})
            print(f"[SKIP] {YEAR}-{month:02d}: {rows:,} rows from existing parquet")
            continue
        except Exception as e:
            print(f"[WARN] Existing file unreadable for {YEAR}-{month:02d}, re-downloading. Error: {e}")

    url = BASE_URL.format(year=YEAR, month=month)
    print(f"\n[GET] {YEAR}-{month:02d}: {url}")

    try:
        resp = requests.get(url, timeout=300, stream=True)
        resp.raise_for_status()

        total_size = int(resp.headers.get("content-length", 0))
        chunk_size = 1024 * 1024  # 1 MB

        data = io.BytesIO()
        with tqdm(total=total_size, unit="B", unit_scale=True, desc=f"  {YEAR}-{month:02d}", leave=False) as pbar:
            for chunk in resp.iter_content(chunk_size=chunk_size):
                if chunk:
                    data.write(chunk)
                    pbar.update(len(chunk))

        data.seek(0)
        with zipfile.ZipFile(data) as zf:
            csv_name = zf.namelist()[0]
            with zf.open(csv_name) as f:
                df = pd.read_csv(f, low_memory=False)

        rows = len(df)
        total_rows += rows
        monthly_counts.append({"year": YEAR, "month": month, "rows": rows, "source": "downloaded"})
        df.to_parquet(out_path, index=False)
        print(f"  [OK] {rows:,} rows saved -> {out_path.name} | Running total: {total_rows:,}")

    except Exception as e:
        failed_months.append(month)
        print(f"  [FAIL] {YEAR}-{month:02d}: {e}")

summary_df = pd.DataFrame(monthly_counts).sort_values("month")
summary_path = RAW_DIR / f"monthly_counts_{YEAR}.csv"
summary_df.to_csv(summary_path, index=False)

print("\n" + "=" * 60)
print(f"Total rows for {YEAR}: {total_rows:,}")
print(f"Summary saved to: {summary_path}")
if failed_months:
    print(f"Failed months: {failed_months}")
    raise RuntimeError(f"Download incomplete for months: {failed_months}")
print("All 12 months available.")
print("=" * 60)

Overall progress:   0%|          | 0/12 [00:00<?, ?it/s]


[GET] 2024-01: https://transtats.bts.gov/PREZIP/On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2024_1.zip


  2024-01:   0%|          | 0.00/27.6M [00:00<?, ?B/s]

  [OK] 547,271 rows saved -> flights_2024_01.parquet | Running total: 547,271

[GET] 2024-02: https://transtats.bts.gov/PREZIP/On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2024_2.zip


  2024-02:   0%|          | 0.00/25.1M [00:00<?, ?B/s]

...

  [OK] 615,497 rows saved -> flights_2024_10.parquet | Running total: 5,913,076

[GET] 2024-11: https://transtats.bts.gov/PREZIP/On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2024_11.zip


  2024-11:   0%|          | 0.00/28.1M [00:00<?, ?B/s]

  [OK] 575,404 rows saved -> flights_2024_11.parquet | Running total: 6,488,480

[GET] 2024-12: https://transtats.bts.gov/PREZIP/On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2024_12.zip


  2024-12:   0%|          | 0.00/30.2M [00:00<?, ?B/s]

  [OK] 590,581 rows saved -> flights_2024_12.parquet | Running total: 7,079,061

Total rows for 2024: 7,079,061
Summary saved to: /Users/hazel/Desktop/26_spring/cis_5450/CIS-5450/data/raw/monthly_counts_2024.csv
All 12 months available.


In [5]:
# Summary table + completeness check
summary = pd.DataFrame(monthly_counts).sort_values('month').reset_index(drop=True)
print(summary.to_string(index=False))

expected_months = set(range(1, 13))
observed_months = set(summary['month'].tolist())
missing_months = sorted(expected_months - observed_months)

print(f"\nTotal: {summary['rows'].sum():,} rows")
print(f"Months covered: {sorted(observed_months)}")
if missing_months:
    raise ValueError(f"Missing months in summary: {missing_months}")
print("Coverage check passed: all months 1-12 present.")

 year  month   rows     source
 2024      1 547271 downloaded
 2024      2 519221 downloaded
 2024      3 591767 downloaded
 2024      4 582185 downloaded
 2024      5 609743 downloaded
 2024      6 611132 downloaded
 2024      7 634613 downloaded
 2024      8 619025 downloaded
 2024      9 582622 downloaded
 2024     10 615497 downloaded
 2024     11 575404 downloaded
 2024     12 590581 downloaded

Total: 7,079,061 rows
Months covered: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]
Coverage check passed: all months 1-12 present.


In [6]:
# Quick peek at columns from January file
sample_path = RAW_DIR / f'flights_{YEAR}_01.parquet'
sample = pd.read_parquet(sample_path)

print(f"Sample file: {sample_path}")
print(f"Columns ({len(sample.columns)}):")
print(sample.columns.tolist())
print(f"\nShape: {sample.shape}")
sample.head(3)

Sample file: /Users/hazel/Desktop/26_spring/cis_5450/CIS-5450/data/raw/flights_2024_01.parquet
Columns (110):
['Year', 'Quarter', 'Month', 'DayofMonth', 'DayOfWeek', 'FlightDate', 'Reporting_Airline', 'DOT_ID_Reporting_Airline', 'IATA_CODE_Reporting_Airline', 'Tail_Number', 'Flight_Number_Reporting_Airline', 'OriginAirportID', 'OriginAirportSeqID', 'OriginCityMarketID', 'Origin', 'OriginCityName', 'OriginState', 'OriginStateFips', 'OriginStateName', 'OriginWac', 'DestAirportID', 'DestAirportSeqID', 'DestCityMarketID', 'Dest', 'DestCityName', 'DestState', 'DestStateFips', 'DestStateName', 'DestWac', 'CRSDepTime', 'DepTime', 'DepDelay', 'DepDelayMinutes', 'DepDel15', 'DepartureDelayGroups', 'DepTimeBlk', 'TaxiOut', 'WheelsOff', 'WheelsOn', 'TaxiIn', 'CRSArrTime', 'ArrTime', 'ArrDelay', 'ArrDelayMinutes', 'ArrDel15', 'ArrivalDelayGroups', 'ArrTimeBlk', 'Cancelled', 'CancellationCode', 'Diverted', 'CRSElapsedTime', 'ActualElapsedTime', 'AirTime', 'Flights', 'Distance', 'DistanceGroup', 'Ca

,Year,Quarter,Month,DayofMonth,DayOfWeek,FlightDate,Reporting_Airline,DOT_ID_Reporting_Airline,IATA_CODE_Reporting_Airline,Tail_Number,...,Div4TailNum,Div5Airport,Div5AirportID,Div5AirportSeqID,Div5WheelsOn,Div5TotalGTime,Div5LongestGTime,Div5WheelsOff,Div5TailNum,Unnamed: 109
0,2024,1,1,8,1,2024-01-08,9E,20363,9E,N485PX,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2024,1,1,9,2,2024-01-09,9E,20363,9E,N912XJ,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2024,1,1,10,3,2024-01-10,9E,20363,9E,N918XJ,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
